# Segmentation Method Comparison

Systematic comparison of Xenium native segmentation vs 
Cellpose-SAM (morphology) vs Cellpose-SAM (H&E).


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cpsam_xenium_analysis import config, data_loader as dl
from cpsam_xenium_analysis.comparison import SegmentationComparator
from cpsam_xenium_analysis.segmentation.post_process import compute_morphology_features

%matplotlib inline


## 1. Load All Segmentation Results


In [ ]:
# Load CPSAM masks
mask_morph = np.load(config.OUTPUT_DIR / 'masks_cpsam_morphology.npy')
mask_he = np.load(config.OUTPUT_DIR / 'masks_cpsam_he.npy')

# Load Xenium cell boundaries as a mask (for comparison)
from cpsam_xenium_analysis.integration import CoordinateAligner
cells_df = dl.load_cells_df()
cell_boundaries = dl.load_cell_boundaries()

print(f'CPSAM Morphology: {len(np.unique(mask_morph))-1} cells')
print(f'CPSAM H&E: {len(np.unique(mask_he))-1} cells')
print(f'Xenium: {len(cells_df)} cells in full dataset')


In [ ]:
# Build Xenium reference mask within the crop ROI
crop = config.CROP_ROI
xs, ys = crop['x_start'], crop['y_start']
xe, ye = xs + crop['width'], ys + crop['height']

# Filter Xenium cells to ROI
aligner = CoordinateAligner()
cells_px = aligner.cell_centroids_to_pixels(cells_df, crop_roi=crop)
print(f'Xenium cells in ROI: {len(cells_px)}')


## 2. Compare Cell Counts and Areas


In [ ]:
comparator = SegmentationComparator()
comparator.add_segmentation('cpsam_morphology', mask_morph)
comparator.add_segmentation('cpsam_he', mask_he)

report = comparator.summary_report()

print('=== Cell Counts ===')
print(report['cell_counts'])
print()
print('=== Area Distribution ===')
print(report['area_distribution'])
print()
print('=== Pairwise Overlap ===')
print(report['pairwise_overlap'])


## 3. Per-Cell Matching Analysis


In [ ]:
comparison = comparator.per_cell_comparison(
    'cpsam_morphology', 'cpsam_he', iou_threshold=0.3
)

print(f'Matched cells: {len(comparison["matching"]["matches"])}')
print(f'Mean IoU: {comparison["mean_iou"]:.3f}')
print(f'Median IoU: {comparison["median_iou"]:.3f}')
print(f'Mean boundary distance: {comparison["mean_boundary_distance"]:.1f} px')
print(f'Precision: {comparison["matching"]["precision"]:.3f}')
print(f'Recall: {comparison["matching"]["recall"]:.3f}')
print(f'F1: {comparison["matching"]["f1"]:.3f}')


## 4. Morphology Feature Comparison


In [ ]:
features_morph = compute_morphology_features(mask_morph)
features_he = compute_morphology_features(mask_he)

df_morph = pd.DataFrame(features_morph).T
df_he = pd.DataFrame(features_he).T

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, col, title in zip(axes.flatten(), 
    ['area', 'circularity', 'eccentricity', 'solidity'],
    ['Cell Area (pixels)', 'Circularity', 'Eccentricity', 'Solidity']
):
    ax.hist(df_morph[col].dropna(), bins=50, alpha=0.6, label='CPSAM Morphology')
    ax.hist(df_he[col].dropna(), bins=50, alpha=0.6, label='CPSAM H&E')
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()


## 5. Comparison Summary


In [ ]:
print('=' * 60)
print('SEGMENTATION COMPARISON SUMMARY')
print('=' * 60)
print()

for method in ['cpsam_morphology', 'cpsam_he']:
    n_cells = len(np.unique(locals()[f'mask_{method.split("_")[1]}'])) - 1
    print(f'{method}: {n_cells} cells')

print()
print(f'Pairwise Dice coeff: {report["pairwise_overlap"].to_string(index=False)}')
print()
print(f'Matched cells (CPSAM Morph vs HE): {len(comparison["matching"]["matches"])}')
print(f'Mean IoU of matched cells: {comparison["mean_iou"]:.3f}')
